In [1]:
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import Counter

In [2]:
emotion_map = {
    "angry": 0,
    "disgust": 1,
    "Fear": 2,
    "happy": 3,
    "neutral": 4,
    "Pleasant_surprise": 5,
    "sad": 6
}

In [3]:
tess_path = r"C:\Users\Thimathi\source\Aurevia\data\raw\TESS"

X = []
y = []

for folder in os.listdir(tess_path):
    folder_path = os.path.join(tess_path, folder)

    if os.path.isdir(folder_path):
        emotion_name = folder.split("_")[-1]

        if emotion_name in emotion_map:
            label = emotion_map[emotion_name]

            for file in os.listdir(folder_path):
                if file.endswith(".wav"):
                    file_path = os.path.join(folder_path, file)

                    audio, sr = librosa.load(file_path, sr=None)

                    # Small noise augmentation
                    audio = audio + 0.005 * np.random.randn(len(audio))

                    # Log-Mel Spectrogram
                    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
                    mel = librosa.power_to_db(mel, ref=np.max)

                    # Pad/trim to fixed size
                    max_len = 128
                    if mel.shape[1] < max_len:
                        pad_width = max_len - mel.shape[1]
                        mel = np.pad(mel, ((0,0),(0,pad_width)), mode='constant')
                    else:
                        mel = mel[:, :max_len]

                    X.append(mel)
                    y.append(label)

X = np.array(X)
y = np.array(y)

print("Total samples:", len(X))
print("Class distribution:", Counter(y))

c:\Users\Thimathi\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total samples: 1000
Class distribution: Counter({np.int64(0): 200, np.int64(1): 200, np.int64(2): 200, np.int64(3): 200, np.int64(4): 200})


In [4]:
X = X[:, np.newaxis, :, :]

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

In [7]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

In [8]:
class TESS_CNN(nn.Module):
    def __init__(self):
        super(TESS_CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 7)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = TESS_CNN()

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

In [10]:
num_epochs = 30
batch_size = 32
best_val_acc = 0
patience = 5
counter = 0

for epoch in range(num_epochs):
    model.train()

    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size]
        yb = y_train[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, preds = torch.max(outputs, 1)
        val_acc = accuracy_score(y_val.numpy(), preds.numpy())

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save(model.state_dict(), "best_tess_model.pth")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered.")
        break

print("Best Validation Accuracy:", best_val_acc)

Epoch 1, Loss: 0.8268, Val Accuracy: 0.7300
Epoch 2, Loss: 0.1394, Val Accuracy: 0.9550
Epoch 3, Loss: 0.0191, Val Accuracy: 0.9900
Epoch 4, Loss: 0.0139, Val Accuracy: 1.0000
Epoch 5, Loss: 0.0073, Val Accuracy: 0.9950
Epoch 6, Loss: 0.0113, Val Accuracy: 1.0000
Epoch 7, Loss: 0.0167, Val Accuracy: 1.0000
Epoch 8, Loss: 0.0004, Val Accuracy: 1.0000
Epoch 9, Loss: 0.0006, Val Accuracy: 1.0000
Early stopping triggered.
Best Validation Accuracy: 1.0
